# Homework09

Exercises with RegEx and scraping

## Goals

- Practice fetching data from the internet
- Practice traversing `html` and `xml` data
- Build our own dataset by combining data from multiple sources

## Import helpers

Run the following cells to import helper functions, files and libraries

In [1]:
!wget -q https://github.com/PSAM-5005-2026S-A/5005-utils/raw/main/src/data_utils.py

In [2]:
import re
import requests

from os import makedirs
from bs4 import BeautifulSoup

from data_utils import image_from_url

## "More Info" Overlay

For this homework we want to create a dataset that can be used to train a model to detect which actors/actresses are on screen at any given time.

Something like the X-Ray feature on Amazon Prime:

<img src="./imgs/amazon-x-ray.webp" height="300px"/>

We'll train the model at a later assignment. For this assignment, what we really need to do is collect some images for some actors and actresses.

### Outline

Here are the steps we'll have to implement in order to achieve this:

1. Pick a movie. Your code should work with any movie, but pick a movie to work with and test your code
2. Go to the movie's tmdb page and copy its url
3. Scrape the name of $5$ actors/actresses from the page's html
4. Scrape some images of each actor/actress from bing/google
5. Save them to a separate subdirectory per person, inside a directory for the movie, for example:
- `hunger-games/` 
   - `elizabeth-banks/`
      - `000_elizabeth-banks.jpg`
      - `001_elizabeth-banks.jpg`
      - ...
   - `jennifer-lawrence/`
      - `000_jennifer-lawrence`
      - `001_jennifer-lawrence`
      - ...

In [17]:
# TODO: movie name and tmdb url
MOVIE_NAME = "The Hunger Games"
TMDB_URL = "https://www.themoviedb.org/movie/70160-the-hunger-games"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}




### Create movie folder

In [18]:
# This will create a folder with the movie name
movie_slug = re.sub(r"[^a-z0-9]+", "-", MOVIE_NAME.lower())
makedirs("./data/images/" + movie_slug, exist_ok=True)

### Scrape Actors/Actresses

In [23]:
# TODO: make a get request to tmdb url
# TODO: load resulting response into a bs object


def get_soup(tmdb_url):
    # TODO: make a GET request to tmdb url
    response = requests.get(tmdb_url, headers=HEADERS)
    
    # Optional (but recommended): check request worked
    response.raise_for_status()

    # TODO: load resulting response into a bs object
    soup = BeautifulSoup(response.text, "html.parser")
    
    return soup

In [30]:
# TODO: Find actors/actresses in html
# NOTE: <li> elements with a "card" class might help

def get_cast(soup, limit=5):
    actors = []

    # TODO: Find actors/actresses in html
    cast_cards = soup.select("li.card")

    for card in cast_cards:
        name_tag = card.select_one("p a")
        if name_tag:
            name = name_tag.text.strip()
            actors.append(name)

        if len(actors) >= limit:
            break

    return actors

In [29]:
# TODO: create subdirectories for each person

def create_actor_dirs(movie_slug, actors):
    # Create main movie directory
    os.makedirs(movie_slug, exist_ok=True)

    for actor in actors:
        actor_slug = slugify(actor)
        actor_path = os.path.join(movie_slug, actor_slug)

        # TODO: create subdirectories for each person
        os.makedirs(actor_path, exist_ok=True)

    return

# TODO HERE

In [28]:
res = requests.get("https://www.bing.com/images/search?q=Timothée+Chalamet&form=HDRSC3&first=2")
bimg_txt = res.text

bing_img_urls = re.findall(r"<img .+? src=[\"\'](https://.+?)[\"\']", bimg_txt)

for img_url in bing_img_urls:
  w,h = re.findall(r"w=([0-9]+)[\s\S]+?h=([0-9]+)", img_url)[0]
  if int(w) > 64 and int(h) > 64:
    print(img_url)

https://tse1.mm.bing.net/th/id/OIP.VBHCvQ4iJ9sxcMpx6iwFnAAAAA?w=158&amp;h=180&amp;c=7&amp;r=0&amp;o=5&amp;pid=1.7
https://tse3.mm.bing.net/th/id/OIP.KNk3VC3moDFeK6Fr7SggbgHaHW?w=147&amp;h=180&amp;c=7&amp;r=0&amp;o=5&amp;pid=1.7
https://tse4.mm.bing.net/th/id/OIP.jiGATQxHxPBLv7QdxOyJNgHaJ3?w=115&h=180&c=7&r=0&o=5&pid=1.7
https://tse2.mm.bing.net/th/id/OIP.8gNekCqK2AyKArFWyvkuoQHaNr?w=115&h=180&c=7&r=0&o=5&pid=1.7
https://tse3.mm.bing.net/th/id/OIP.vS0_4ve2XVtX0CtpPTbdzgHaEK?w=267&h=180&c=7&r=0&o=5&pid=1.7
https://tse3.mm.bing.net/th/id/OIP.ilmmmYLTjZB-cZyk_AA6nwHaEK?w=211&h=180&c=7&r=0&o=5&pid=1.7
https://tse4.mm.bing.net/th/id/OIP.oKeoKbLmm8UarpeES3E2yQHaFb?w=89&h=89&c=1&rs=1&qlt=90&r=0&pid=InlineBlock
https://tse4.mm.bing.net/th/id/OIP.MUDOxejJsw5xBxuq1ndb9wHaE7?w=89&h=90&c=1&rs=1&qlt=90&r=0&pid=InlineBlock
https://tse3.mm.bing.net/th/id/OIP.HUfYL1uUp8kcRJPmVw0S0gHaDt?w=255&h=180&c=7&r=0&o=5&pid=1.7
https://tse2.mm.bing.net/th/id/OIP.Zd6fLZVRJGwZQc8gnVfqEwHaLH?w=89&h=89&c=1&rs=1&qlt=9